In [4]:
%pip install yfiles_jupyter_graphs --quiet
%pip install pygraphviz --quiet

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [17]:
!git status

On branch yfiles-main
Your branch is up to date with 'origin/yfiles-main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   .ipynb_checkpoints/yFiles_4FFBO-checkpoint.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore

no changes added to commit (use "git add" and/or "git commit -a")


In [19]:
!git push -u origin yfiles-main

Branch 'yfiles-main' set up to track remote branch 'yfiles-main' from 'origin'.
Everything up-to-date


In [23]:
!git --version 
!git remote -v

git version 2.34.1
origin	https://github.com/sgarnell/archive.git (fetch)
origin	https://github.com/sgarnell/archive.git (push)


In [81]:

# Stage changes
!git add yFiles_4FFBO.ipynb

# Commit the new changes
!git commit -m "Latest version of Graph Notebook"

# Push to remote
!git push origin yfiles-main


[yfiles-main 0ba1e4a] Latest version of Graph Notebook
 1 file changed, 35 insertions(+), 13 deletions(-)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 16 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 870 bytes | 870.00 KiB/s, done.
Total 3 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/sgarnell/archive.git
   01d62b6..0ba1e4a  yfiles-main -> yfiles-main


In [30]:
# Step 1: Fetch the latest changes from GitHub
!git checkout origin/main -- filtered_output.gv
!git fetch origin
!git diff origin/main filtered_output.gv || true; [ $? -eq 0 ] && echo "🟢 No changes detected"

🟢 No changes detected


In [107]:
%pdb on
import pdb

Automatic pdb calling has been turned ON


In [151]:
import sys
print(sys.executable)
import re
import time
from pygraphviz import AGraph
from yfiles_jupyter_graphs import GraphWidget

/opt/conda/bin/python


In [10]:
!curl -o filtered_output.gv https://raw.githubusercontent.com/sgarnell/archive/yfiles-main/filtered_output.gv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 74811  100 74811    0     0   238k      0 --:--:-- --:--:-- --:--:--  238k


In [5]:
try:
  import google.colab
  from google.colab import output
  output.enable_custom_widget_manager()
except:
  pass



In [77]:
def _node_color_mapping(node):
    label = node.get("properties", {}).get("label", "")
    if "cent=" in label:
        return "red"  # neuron
    return "cyan"    # synapse or other


def _node_size_mapping(node):
    label = node.get("properties", {}).get("label", "")
    yf_lable = node.get("properties", {}).get("yf_label", "")
    if "cent=" in label:
        return (100.0, 100.0)
    if "--" in yf_lable:
        return (15.0, 15.0)
    else:
        return (60.0, 60.0)  # fallback height if neither condition matches


group_colors = {
    "FB5": "#FFCCCC",
    "DPM": "#CCFFCC",
    "APL": "#CCCCFF",
    # Add more as needed
}


def _node_parent_group_mapping(node):
    label = node.get("properties", {}).get("label", "")
    if "cent=" in label:
        group_id = label[:3]
        return group_id
    return None


color_name_to_hex = {
    'red': '#FF0000',
    'orangered3': '#CD3700',     # X11 approximation
    'orangered2': '#EE4000',     # X11 approximation
    'gold': '#FFD700',
    'chartreuse': '#7FFF00',
    'limegreen': '#32CD32',
    'lime': '#00FF00',
    'cyan3': '#00CDCD',          # X11 approximation
    'blue': '#0000FF'
}


# def _custom_edge_color_mapping(edge):
    
#     edgeColorName = edge.get("properties", {}).get("color", "")
#     return (color_name_to_hex[edgeColorName])

def _custom_factor_mapping(edge):
    penwidth = edge.get("properties", {}).get("penwidth", "")
    try:
        pw = float(penwidth)
    except (ValueError, TypeError):
        return 1.0  # fallback for missing or invalid penwidth

    # Define expected penwidth range from your dataset
    min_pw, max_pw = 1.0, 3.0  # adjust if your data has wider spread
    min_factor, max_factor = 1.0, 10.0

    # Clamp penwidth to expected range
    pw_clamped = max(min_pw, min(pw, max_pw))

    # Linear interpolation
    scale = (pw_clamped - min_pw) / (max_pw - min_pw)
    thickness = min_factor + scale * (max_factor - min_factor)

    return round(thickness, 2)


In [142]:
def _widget(graph):

    # Inject a label basee on the node ID when labels are blank
    for node in graph.nodes():
        label = node.attr.get("label", "")
        nodeName = node.name
        if not label:
            node.attr["label"] = node.name  # or node.get_name() if that works
    
    w = GraphWidget(graph=graph)

    w.set_node_color_mapping(_node_color_mapping)
    w.set_node_size_mapping(_node_size_mapping)
    w.set_node_parent_group_mapping(_node_parent_group_mapping)
    w.set_edge_color_mapping(_custom_edge_color_mapping)
    w.set_edge_thickness_factor_mapping(_custom_factor_mapping)
    w.hierarchic_layout()

    return w


In [143]:
# Step 1: Load the Graphviz file using pygraphviz
graph = AGraph("filtered_output.gv")


In [144]:
w = _widget(graph)
w

GraphWidget(layout=Layout(height='800px', width='100%'))

In [196]:
propagate_from(w, "ExR5(ring)")

In [195]:
highlighted_nodes = set()
highlighted_edges = set()

label_to_id = {
    node["properties"].get("yf_label", node["label"]): node["id"]
    for node in w.nodes
}


def custom_edge_styles_mapping(edge):
    if edge.get("id") in highlighted_edges:
        return {"color": "orange", "strokeWidth": 2}
    return {"color": "gray", "strokeWidth": 1}

def custom_node_styles_mapping(node):
    node_id = node.get("id") or node.get("name")
    pdb.set_trace()
    if node_id in highlighted_nodes:
        return {"color": "red", "shape": "ellipse"}
    return {"color": "lightgray", "shape": "ellipse"}

def propagate_from(graph_widget, node_label):
    node_id = label_to_id.get(node_label)
    if node_id is None:
        print(f"Node label '{node_label}' not found.")
        return

    highlighted_nodes.add(node_id)
    graph_widget.set_node_styles_mapping(custom_node_styles_mapping)
    time.sleep(1)

    outgoing = [e for e in graph_widget.edges if e["start"] == node_id]
    for edge in outgoing:
        highlighted_edges.add(edge["id"])
    graph_widget.set_edge_styles_mapping(custom_edge_styles_mapping)
    time.sleep(1)

    for edge in outgoing:
        highlighted_nodes.add(edge["end"])
    graph_widget.set_node_styles_mapping(custom_node_styles_mapping)




In [175]:
w.set_node_styles_mapping(custom_node_styles_mapping)
w.set_edge_styles_mapping(custom_edge_styles_mapping)

In [184]:
for e in w.nodes[:5]:
    print(e)

{'id': 0, 'properties': {'color': 'brown2', 'fixedsize': 'true', 'fontcolor': 'white', 'fontsize': '5', 'height': '0.1', 'label': 'APL cent= 0.1', 'shape': 'circle', 'style': 'filled', 'yf_label': 'APL'}, 'parentId': 'group#APL', 'color': 'red', 'styles': {}, 'label': 'APL cent= 0.1', 'scale_factor': 1.0, 'type': 'red', 'size': (100.0, 100.0), 'position': (0.0, 0.0)}
{'id': 1, 'properties': {'color': 'brown2', 'fixedsize': 'true', 'fontcolor': 'white', 'fontsize': '5', 'height': '0.1', 'label': 'CRE042(ADM10) cent= 0.1', 'shape': 'circle', 'style': 'filled', 'yf_label': 'CRE042(ADM10)'}, 'parentId': 'group#CRE', 'color': 'red', 'styles': {}, 'label': 'CRE042(ADM10) cent= 0.1', 'scale_factor': 1.0, 'type': 'red', 'size': (100.0, 100.0), 'position': (0.0, 0.0)}
{'id': 2, 'properties': {'color': 'brown2', 'fixedsize': 'true', 'fontcolor': 'white', 'fontsize': '5', 'height': '0.1', 'label': 'CRE043_b(ADM10) cent= 0.1', 'shape': 'circle', 'style': 'filled', 'yf_label': 'CRE043_b(ADM10)'}, '